# **9. 데이터 분석을 위한 pandas 3**

## **학습 목표**

- 정렬하고 파생변수를 만든다. (`sort_values`, `rank`·`cumsum`, `rename`·`drop`·`map`·`replace`, `pd.cut`, `apply`)
- 두 표를 합친다. (`merge(how="left"·"inner")`, `concat(axis=0·1)`)
- 묶어서 집계하고 표의 모양을 바꾼다. (`groupby`·`agg`, `pivot_table`, `melt`)
- 결과를 CSV·Excel 파일로 저장한다.


---

# **0. 라이브러리와 데이터 준비**




In [ ]:
import os
import numpy as np
import pandas as pd


from google.colab import drive

# Google Drive를 Colab에 연결한다. 실행하면 권한 승인 창이 나타난다 → "Google Drive에 연결" → 계정 선택 → "허용"
drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"

print("데이터 폴더:", data_folder)
print("data 폴더의 파일:", os.listdir(data_folder))


Mounted at /content/drive
데이터 폴더: /content/drive/MyDrive/HD-AI/data
data 폴더의 파일: ['FS1.txt', 'PS1.txt', 'TS1.txt', 'profile.txt', 'PdM_maint.csv', 'PdM_telemetry.csv', 'ai4i2020.csv', 'secom.names', 'secom.data', 'secom_labels.data', 'MetroPT3(AirCompressor).csv', 'psych_health_education_dataset.csv', 'Steel_industry_data.csv', 'superstore.csv', 'PdM_telemetry_dirty.csv', 'PdM_failures_dirty.csv', 'PdM_errors_dirty.csv', 'PdM_machines_dirty.csv', 'PdM_errors.csv', 'PdM_machines.csv', 'PdM_failures.csv', 'loan_data.csv', 'penguins.csv', 'mpg.csv', 'tips.csv', 'Bank_customers_data.csv', 'titanic.csv', 'ai4i.db', 'ai4i_dirty.csv', 'superstore.xlsx', 'manufacturing_clean.csv', 'manufacturing_practice_result.csv', 'superstore_요약.xlsx', '매출_보고서.xlsx', '지역별_제품군_매출.png', '지역별_매출.png', '설비_고장목록.csv', 'paired_measurement.csv', 'auto_mpg.csv', 'Steel_기술통계표.xlsx', 'Usage_kWh_히스토그램.png', 'Lagging_Current_Reactive.Power_kVarh_히스토그램.png', 'CO

In [ ]:
# sales.csv 를 불러와 작업용 df 를 만든다.
csv_path = os.path.join(data_folder, "sales.csv")

df_csv = pd.read_csv(csv_path)
df = df_csv.copy()
print(df.dtypes)

df["주문일"] = pd.to_datetime(df["주문일"])   # DAY 4 2절: 문자열 날짜 → 날짜형 (기간 조회·정렬에 필요)

print("데이터 크기:", df.shape)
print(df.dtypes)
df.head()

주문번호    object
주문일     object
지역      object
제품군     object
제품명     object
수량       int64
매출       int64
dtype: object
데이터 크기: (15, 7)
주문번호            object
주문일     datetime64[ns]
지역              object
제품군             object
제품명             object
수량               int64
매출               int64
dtype: object


,주문번호,주문일,지역,제품군,제품명,수량,매출
0,O001,2026-01-05,서울,전자,노트북,2,2400000
1,O002,2026-01-12,부산,가구,책상,3,900000
2,O003,2026-01-20,서울,식품,커피,10,300000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
4,O005,2026-02-14,부산,식품,간식세트,8,480000


In [ ]:
# superstore 주문상품 기록을 불러온다. 뒤의 실습에서 orders 라는 이름으로 사용한다.
store_path = os.path.join(data_folder, "superstore.xlsx")
orders = pd.read_excel(store_path, sheet_name="Orders")
returns = pd.read_excel(store_path, sheet_name="Returns")
print("Returns 크기:", returns.shape)

print("orders크기:", orders.shape)                 # (10194, 21)
print("orders결측치가 있는 열:")
print(orders.isna().sum()[orders.isna().sum() > 0])   # 0보다 큰 열만
orders.head(3)


Returns 크기: (296, 2)
orders크기: (10194, 21)
orders결측치가 있는 열:
Series([], dtype: int64)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717


# **1. 정렬과 파생변수 만들기**



## **1) `sort_values()`로 정렬하기**

지정한 열을 기준으로 행의 순서를 바꾼다.

```python
df.sort_values("열이름", ascending=False)
```

`ascending`은 정렬 방향을 정한다. 생략하면 오름차순이 기본이다.

| 지정값 | 정렬 방향 |
|---|---|
| `ascending=True` (기본값) | 작은 값부터 (오름차순) |
| `ascending=False` | 큰 값부터 (내림차순) |

기준이 여러 개일 때는 열 이름을 리스트로 전달한다. 앞에 적은 열이 먼저 적용되고, 값이 같을 때 다음 열로 순서를 정한다. 정렬 방향도 열의 개수만큼 리스트로 지정한다.

```python
df.sort_values(["지역", "매출"], ascending=[True, False])
```

위 코드는 지역을 가나다순으로 정렬하고, 같은 지역 안에서는 매출이 높은 순서로 정렬한다.

정렬은 원본을 바꾸지 않고 정렬된 결과를 새로 돌려준다.

공식 문서: [sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)

정렬 기준 열에 결측값이 있다면 `na_position="first"` 또는 `na_position="last"`로 결측값의 위치를 지정할 수 있다. 기본값은 `"last"`이다.


In [ ]:
# [예제] 매출 내림차순 정렬
df.sort_values("매출", ascending=False)


,주문번호,주문일,지역,제품군,제품명,수량,매출
13,O014,2026-05-03,서울,전자,스마트폰,4,3200000
0,O001,2026-01-05,서울,전자,노트북,2,2400000
8,O009,2026-03-19,서울,전자,태블릿,3,2100000
10,O011,2026-04-04,부산,전자,이어폰,9,1350000
3,O004,2026-02-03,대전,전자,모니터,4,1200000
14,O015,2026-05-18,부산,가구,소파,1,1100000
1,O002,2026-01-12,부산,가구,책상,3,900000
7,O008,2026-03-11,부산,가구,수납장,2,800000
5,O006,2026-02-25,서울,가구,의자,5,750000
11,O012,2026-04-15,서울,가구,테이블,2,700000


In [ ]:
import pandas as pd
import numpy as np

# 1. 결측값(NaN)이 포함된 데이터 만들기
df = pd.DataFrame({
    "지역": ["서울", "부산", "서울", "부산", "대구"],
    "매출": [300, 200, np.nan, 400, 100]
})

print("원본 데이터:")
display(df)


# 2. 다중 정렬
# 지역은 가나다순(오름차순), 같은 지역 안에서는 매출 높은 순(내림차순)
print("지역 오름차순, 매출 내림차순:")
display(df.sort_values(["지역", "매출"], ascending=[True, False]))


# 3. na_position="first" : 결측값을 맨 위에 배치
# 매출은 높은 순으로 정렬하되, NaN은 가장 위에 놓는다.
print("결측값을 맨 위로:")
display(df.sort_values("매출", ascending=False, na_position="first"))


# 4. na_position="last" : 결측값을 맨 아래에 배치 (기본값)
# 매출은 높은 순으로 정렬하되, NaN은 가장 아래에 놓는다.
print("결측값을 맨 아래로:")
display(df.sort_values("매출", ascending=False, na_position="last"))

원본 데이터:


,지역,매출
0,서울,300.0
1,부산,200.0
2,서울,NaN
3,부산,400.0
4,대구,100.0


지역 오름차순, 매출 내림차순:


,지역,매출
4,대구,100.0
3,부산,400.0
1,부산,200.0
0,서울,300.0
2,서울,NaN


결측값을 맨 위로:


,지역,매출
2,서울,NaN
3,부산,400.0
0,서울,300.0
1,부산,200.0
4,대구,100.0


결측값을 맨 아래로:


,지역,매출
3,부산,400.0
0,서울,300.0
1,부산,200.0
4,대구,100.0
2,서울,NaN


## **실습 1: 정렬**

`df`는 바로 앞의 정렬 예제에서 만든 **결측값이 포함된 5행짜리 표**이다 (`지역`, `매출` 열). 이 `df`를 사용해 다음을 수행한다.


In [ ]:
# [실습] 1: 정렬
# 1. 매출이 큰 순서대로 정렬한 상위 3개 행

# 2. 매출이 작은 순서대로 정렬

# 3. 지역은 오름차순, 같은 지역에서는 매출 내림차순


In [ ]:
# [실습] 정렬 시 결측값 위치
# 1. 매출을 내림차순으로 정렬하고, 결측값은 맨 앞으로 보내시오.


# 2. 매출을 오름차순으로 정렬하고, 결측값은 맨 뒤로 보내시오.


# 3. 두 결과의 처음 3개 행을 비교하시오.


## **2) 새로운 열 만들기**

기존 열을 계산해 새로운 열을 만들 수 있다. 이렇게 만든 열을 파생변수라고 한다.

```python
df["새열이름"] = 계산식
```

없던 이름을 지정하면 새 열이 만들어지고, 이미 있는 이름을 지정하면 값이 바뀐다. 매출 비중, 단가, 등급처럼 원본에 없는 지표를 만들 때 사용한다.

새 열을 만드는 코드는 원본 `df`를 직접 바꾼다는 점에 주의한다.

In [ ]:
# [예제] 계산 결과로 새 열 만들기
# 전체 매출에서 각 행이 차지하는 비중
df["매출비중"] = df["매출"] / df["매출"].sum()

# 조건에 따라 값이 달라지는 열
df["고액여부"] = df["매출"] >= 300

df.head()


,지역,매출,매출비중,고액여부
0,서울,300.0,0.3,True
1,부산,200.0,0.2,False
2,서울,NaN,NaN,False
3,부산,400.0,0.4,True
4,대구,100.0,0.1,False


## **3) 인덱스 정리 (`reset_index`)**

조건으로 데이터를 걸러내면 원래의 인덱스 번호가 그대로 남아 `0, 3, 7`처럼 띄엄띄엄 보인다. 인덱스를 0부터 다시 매기려면 `reset_index()`를 사용한다.

```python
결과.reset_index(drop=True)
```

`drop=True`를 지정하지 않으면 기존 인덱스가 새로운 열로 추가되므로, 보통 함께 지정한다.

공식 문서: [reset_index()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html)


In [ ]:
# [예제] 띄엄띄엄한 인덱스를 0부터 다시 매기기
result = df[df["매출"] >= 100]
print("정리 전 인덱스:", list(result.index))

result = result.reset_index(drop=True)
print("정리 후 인덱스:", list(result.index))


정리 전 인덱스: [0, 1, 3, 4]
정리 후 인덱스: [0, 1, 2, 3]


In [ ]:

import pandas as pd
from IPython.display import display

# [예제] reset_index()와 drop 옵션 비교

# 1. 매출 데이터 만들기
df = pd.DataFrame({
    "지역": ["서울", "부산", "대구", "인천", "광주"],
    "매출": [1200000, 750000, 1500000, 900000, 2100000]
})

display(df)



,지역,매출
0,서울,1200000
1,부산,750000
2,대구,1500000
3,인천,900000
4,광주,2100000


In [ ]:


# 2. 매출 100만 원 이상인 데이터 필터링
# 조건에 맞는 행을 선택해도 기존 인덱스는 유지된다.
result = df[df["매출"] >= 1000000]

print("필터링 후 인덱스:", result.index.tolist())
# 출력: [0, 2, 4]

display(result)



필터링 후 인덱스: [0, 2, 4]


,지역,매출
0,서울,1200000
2,대구,1500000
4,광주,2100000


In [ ]:

# 3. drop=True : 기존 인덱스를 버리고 새 인덱스 생성
result_clean = result.reset_index(drop=True)

print("drop=True 새 인덱스:", result_clean.index.tolist())
# 출력: [0, 1, 2]

display(result_clean)



drop=True 새 인덱스: [0, 1, 2]


,지역,매출
0,서울,1200000
1,대구,1500000
2,광주,2100000


In [ ]:

# 4. drop=False : 기존 인덱스를 일반 열로 보존하고 새 인덱스 생성
result_kept = result.reset_index(drop=False)

print("drop=False 새 인덱스:", result_kept.index.tolist())
# 출력: [0, 1, 2]

display(result_kept)


drop=False 새 인덱스: [0, 1, 2]


,index,지역,매출
0,0,서울,1200000
1,2,대구,1500000
2,4,광주,2100000


In [ ]:


# 6. 기존 인덱스 조회
# drop=False로 기존 인덱스가 'index'라는 일반 열에 저장되었다.
print("기존 인덱스:", result_kept["index"].tolist())
# 출력: [0, 2, 4]

기존 인덱스: [0, 2, 4]


### **[실습] 인덱스 다시 매기기**

In [ ]:
# 실습 데이터 준비 (실행 후 문제를 푸시오)
index_sample = pd.DataFrame({"지역": ["서울", "부산", "대구", "인천"], "매출": [150, 90, 220, 70]})
display(index_sample)

,지역,매출
0,서울,150
1,부산,90
2,대구,220
3,인천,70


In [ ]:
# [실습] reset_index()
# 1. 아래 표에서 매출 100 이상인 행만 선택하시오.
# 2. 선택 결과의 인덱스를 0부터 다시 매기시오.
# 3. reset_index(drop=True) 결과를 출력하시오.


## **실습 2: 파생변수 만들기**

`df_csv`를 복사한 `practice_df`에 다음 열을 새로 만든다. 새 열은 `practice_df["새열이름"] = 계산식` 형태로 만든다.

1. `매출_만원`: `매출`을 10000으로 나눈 값 (예: 2400000 → 240.0)
2. `단가`: `매출`을 `수량`으로 나눈 값 (예: 2400000 ÷ 2 → 1200000.0)
3. `대량주문`: `수량 >= 5` 조건식의 결과. 5개 이상이면 `True`, 아니면 `False`
4. `list(practice_df.columns)`로 세 열이 추가되었는지 확인하고, `practice_df.head()`로 값을 본다


In [ ]:
# 실습 2: 파생변수 만들기
practice_df = df_csv.copy()


## **4) 순위 `rank()`와 누적 합 `cumsum()`**

`rank()`는 값의 크기로 순위를 매기고, `cumsum()`은 첫 번째 값부터 현재 값까지 차례로 더한 누적 합을 구한다.

```python
df["매출순위"] = df["매출"].rank(ascending=False, method="min").astype(int)
# ascending=False: 큰 값이 1위 / method="min": 같은 값은 같은 순위
```

| 옵션 | 뜻 | 예 |
|---|---|---|
| `ascending` | `True` 작은 값이 1위(기본) · `False` 큰 값이 1위 | `rank(ascending=False)` |
| `method` | 같은 값의 순위 처리. `"average"`(기본, 동점 순위의 평균, 예: 2.5) · `"min"`(같은 순위, 다음 순위는 건너뜀) · `"dense"`(건너뛰지 않음) · `"first"`(먼저 나온 행이 앞) | `rank(method="min")` |

점수 `[90, 80, 80, 70]`에 큰 값부터 순위를 매기면 `method`에 따라 다음과 같다.

```text
 값          90   80   80   70
 average      1   2.5  2.5   4
 min          1    2    2    4     (3위를 건너뜀)
 dense        1    2    2    3     (건너뛰지 않음)
 first        1    2    3    4     (먼저 나온 80이 앞)
```

`rank()`의 결과는 `2.5`처럼 소수가 나올 수 있어 실수형이다. 결측값이 없는 순위에서 정수로 보려면 `astype(int)`를 붙인다. 결측 순위가 `NaN`이면 일반 정수형 변환에서 오류가 날 수 있으므로 `Int64` 같은 결측값 지원 정수형을 사용하거나 결측값을 먼저 처리한다.

`cumsum()`은 **현재 행 순서대로** 더한다. 날짜순 누적처럼 순서가 중요하면 먼저 정렬한다. 마지막 누적값은 전체 합계 `sum()`과 같다.

```python
by_date = df.sort_values("주문일")
by_date["누적매출"] = by_date["매출"].cumsum()
```

결측값이 있으면 그 자리는 `NaN`으로 두고 다음 행부터 누적을 이어 간다(`skipna=True`, 기본값). `skipna=False`면 결측값 이후가 모두 `NaN`이 된다.

공식 문서: [Series.rank()](https://pandas.pydata.org/docs/reference/api/pandas.Series.rank.html) · [Series.cumsum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.cumsum.html)

In [ ]:
# [예제] 매출 순위와 누적 매출
# 앞선 reset_index() 예제에서 df를 다른 표로 변경했으므로 원본 매출 데이터를 복원한다.
df = df_csv.copy()
df["주문일"] = pd.to_datetime(df["주문일"])

ranked = df[["주문번호", "주문일", "제품명", "매출"]].copy()     # 연습용 복사본

ranked["매출순위"] = ranked["매출"].rank(ascending=False, method="min").astype(int)   # 큰 매출이 1위
display(ranked.sort_values("매출순위").head(5))                 # 1~5위


,주문번호,주문일,제품명,매출,매출순위
13,O014,2026-05-03,스마트폰,3200000,1
0,O001,2026-01-05,노트북,2400000,2
8,O009,2026-03-19,태블릿,2100000,3
10,O011,2026-04-04,이어폰,1350000,4
3,O004,2026-02-03,모니터,1200000,5


In [ ]:

by_date = ranked.sort_values("주문일")                          # 누적은 순서가 중요하므로 날짜순으로 정렬
by_date["누적매출"] = by_date["매출"].cumsum()                   # 위에서부터 차례로 더한다
display(by_date.head(4))

print(by_date["누적매출"].iloc[-1]) #마지막누적값
print(ranked["매출"].sum()) #전체합계


,주문번호,주문일,제품명,매출,매출순위,누적매출
0,O001,2026-01-05,노트북,2400000,2,2400000
1,O002,2026-01-12,책상,900000,7,3300000
2,O003,2026-01-20,커피,300000,15,3600000
3,O004,2026-02-03,모니터,1200000,5,4800000


16810000
16810000


In [ ]:
# 같은 값의 순위 처리 비교
score = pd.Series([90, 80, 80, 70])
for m in ["average", "min", "dense", "first"]:
    print(m, score.rank(ascending=False, method=m).tolist())


average [1.0, 2.5, 2.5, 4.0]
min [1.0, 2.0, 2.0, 4.0]
dense [1.0, 2.0, 2.0, 3.0]
first [1.0, 2.0, 3.0, 4.0]


In [ ]:
# [실습] rank() 와 cumsum()
# 문제: 일별 생산량 표에서 생산량 순위(큰 값 1위, 같은 값은 같은 순위)를 매기고, 날짜순 누적 생산량을 구한다.
prod = pd.DataFrame({"날짜": pd.to_datetime(["2026-03-04", "2026-03-02", "2026-03-05", "2026-03-03", "2026-03-06"]),
                     "생산량": [130, 120, 145, 130, 110 ]})

# prod = pd.DataFrame({"날짜": pd.to_datetime(["2026-03-04", "2026-03-02", "2026-03-05", "2026-03-03", "2026-03-06", "2026-03-07"]),
#                      "생산량": [130, 120, 145, 130, 110 , np.nan]})


# 2. 생산량이 큰 순서로 method="min" 순위를 정수로 바꿔 "순위" 열에 넣고 리스트로 출력한다.
# 출력: [2, 4, 1, 2, 5]


# 3. 날짜순으로 정렬해 prod 에 다시 담고 "누적생산량" 열을 만든 뒤 표를 표시한다.


# 4. 마지막 누적생산량이 생산량 합계와 같은지 비교해 출력한다.
# 출력: True

# 5. 누적생산량이 처음으로 400 이상이 되는 날짜를 출력한다.
# 출력: 2026-03-05


## **5) 열 이름 바꾸기와 열 삭제 (`rename`, `drop`)**

파일은 열 이름이 영문이거나 `Air temperature [K]`처럼 길어서 코드가 읽기 어렵다. `rename()`으로 바꾼다. 딕셔너리에 `{"원래이름": "새이름"}`을 적는다.

```python
df = df.rename(columns={"Sales": "매출", "Quantity": "수량"})
```

필요 없는 열은 `drop(columns=[...])`으로 지운다. 두 메서드 모두 바뀐 결과를 돌려줄 뿐 원본은 그대로이므로 다시 대입한다.

```python
df = df.drop(columns=["매출비중", "고액여부"])
```

열 이름 전체를 한 번에 바꾸려면 `df.columns = [...]`에 새 이름 목록을 넣는다. 열 개수와 목록 길이가 같아야 한다.

공식 문서: [rename()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html) · [drop()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)


In [ ]:
# [예제] 열 이름 바꾸기와 삭제
renamed = df.rename(columns={"제품명": "품목", "매출": "매출액"})
print(list(renamed.columns))

dropped = df.drop(columns=["제품군"])
print(list(dropped.columns))
print(list(df.columns))      # 원본은 그대로다


['주문번호', '주문일', '지역', '제품군', '품목', '수량', '매출액']
['주문번호', '주문일', '지역', '제품명', '수량', '매출']
['주문번호', '주문일', '지역', '제품군', '제품명', '수량', '매출']


### **[실습] 열 이름 변경과 삭제**

In [ ]:
# 실습 데이터 준비 (실행 후 문제를 푸시오)
column_sample = pd.DataFrame({"품목": ["볼트", "너트"], "수량": [10, 20], "담당자": ["김", "이"]})
display(column_sample)

,품목,수량,담당자
0,볼트,10,김
1,너트,20,이


In [ ]:
# [실습] rename()과 drop()
# 1. 연습용 표의 "수량" 열 이름을 "판매수량"으로 바꾸시오.
# 2. 바꾼 표에서 "담당자" 열을 제외하여 출력하시오.
# 3. 원본 연습용 표의 열 이름이 유지되는지 확인하시오.


## **6) 값 바꾸기 `map()`**

`map()`은 **열에 들어 있는 값을 다른 값으로 바꿀 때** 사용한다.

예를 들어 지역 이름을 권역 이름으로 변경할 수 있다.

- `"서울"` → `"수도권"`
- `"부산"` → `"영남"`
- `"대전"` → `"충청"`

딕셔너리에 **기존 값과 변경할 값을 짝지어** 작성한다.

```python
df["권역"] = df["지역"].map({
    "서울": "수도권",
    "부산": "영남",
    "대전": "충청"
})
```

`map()`은 변환한 결과를 반환하므로, 위 코드에서는 새로운 `"권역"` 열에 저장한다. 기존 `"지역"` 열은 그대로 유지된다.

### **`map()`과 `rename()`의 차이**

| 기능 | 바꾸는 대상 | 설명 |
|---|---|---|
| `map()` | 열 안에 있는 값 | `"서울"` → `"수도권"` |
| `rename()` | 열 이름 | `"Sales"` → `"매출"` |

- `map()` : **데이터 값 변경**
- `rename()` : **열 이름 변경**

### **주의: 딕셔너리에 없는 값**

`map()`에 전달한 딕셔너리에 없는 값은 `NaN`(결측값)으로 표시된다.

예를 들어 `"인천"`이 딕셔너리에 없으면 변환 결과는 `NaN`이 된다.

```python
df["권역"].isna().sum()
# 변환 후 결측값 개수 확인
```

따라서 변환할 값이 딕셔너리에 모두 포함되어 있는지 확인하는 것이 좋다.

공식 문서: [Pandas Series.map()](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)

In [ ]:
import pandas as pd

# 1. 지역 데이터 만들기
df = pd.DataFrame({
    "지역": ["서울", "부산", "대전", "인천", "서울"]
})

print("원본 데이터:")
display(df)


# 2. 지역을 권역으로 변경할 딕셔너리 만들기
region_map = {
    "서울": "수도권",
    "부산": "영남",
    "대전": "충청"
}

# 3. map()으로 지역 값을 변경하여 새 열에 저장
# 딕셔너리에 없는 "인천"은 NaN으로 표시된다.
df["권역"] = df["지역"].map(region_map)

print("권역 변환 결과:")
display(df)


# 4. 권역별 데이터 개수 확인
print("권역별 건수:")
print(df["권역"].value_counts())


# 5. 변환되지 않은 값(NaN) 개수 확인
print("빠진 값:", df["권역"].isna().sum())
# 1

원본 데이터:


,지역
0,서울
1,부산
2,대전
3,인천
4,서울


권역 변환 결과:


,지역,권역
0,서울,수도권
1,부산,영남
2,대전,충청
3,인천,NaN
4,서울,수도권


권역별 건수:
권역
수도권    2
영남     1
충청     1
Name: count, dtype: int64
빠진 값: 1


In [ ]:
# [실습] map(): 지정되지 않은 값 확인
# orders의 "Ship Mode" 열을 한국어로 변환하시오.

# [배송 방식 변환 기준]
# Standard Class → 일반
# Second Class   → 우선
# First Class    → 특급
# Same Day       → 당일

# 1. "Ship Mode" 열의 배송 방식별 건수를 확인하시오.
#    value_counts()를 사용한다.

# 2. 배송 방식 4개 중 "Same Day"를 제외한 3개만
#    딕셔너리 ship_map에 작성하시오.

# 3. map()을 적용한 결과를 ship_korean 변수에 저장하고,
#    변환되지 않아 NaN이 된 값의 개수를 출력하시오.
#    예상 결과: 빠진 값: 547

# 4. ship_map에 "Same Day": "당일"을 추가하시오.

# 5. orders를 복사하여 orders_ship을 만들고,
#    map()으로 변환한 결과를 "배송" 열에 저장하시오.

# 6. "배송" 열의 값별 건수와 결측값 개수를 출력하시오.
#    예상 결과: 일반 6120, 우선 1979, 특급 1548, 당일 547
#    예상 결과: 빠진 값: 0

## **7) 적힌 값만 바꾸기 `replace()`**

`map()`과 `replace()`는 둘 다 딕셔너리로 값을 바꾸지만, 딕셔너리에 **없는 값**을 다르게 처리한다.

| 메서드 | 딕셔너리에 있는 값 | 딕셔너리에 없는 값 | 쓰는 상황 |
|---|---|---|---|
| `map()` | 바뀐다 | `NaN`이 된다 | 모든 값을 새 이름으로 1:1 대응할 때 (코드 → 이름) |
| `replace()` | 바뀐다 | 원래 값 그대로 | 일부 값만 고칠 때 (오타·표기 통일) |

```python
df["지역"].map({"서울": "수도권"})       # 부산·대전은 NaN
df["지역"].replace({"서울": "수도권"})   # 부산·대전은 그대로
```


공식 문서: [Series.replace()](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html) · [Series.map()](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)

In [ ]:
import pandas as pd

# 1. 지역 데이터 만들기
df = pd.DataFrame({
    "지역": ["서울", "부산", "대구", "서울"]
})

print("원본 데이터:")
print(df["지역"].tolist())


# 2. map(): {"서울": "수도권"}딕셔너리에 없는 값은 NaN으로 변경
print("map 결과:")
print(df["지역"].map({"서울": "수도권"}).tolist())


# 3. replace(): 딕셔너리에 없는 값은 그대로 유지
print("replace 결과:")
print(df["지역"].replace({"서울": "수도권"}).tolist())

원본 데이터:
['서울', '부산', '대구', '서울']
map 결과:
['수도권', nan, nan, '수도권']
replace 결과:
['수도권', '부산', '대구', '수도권']


### **[실습] map과 replace 비교**

In [ ]:
# 실습 데이터 준비 (실행 후 문제를 푸시오)
status_sample = pd.DataFrame({"상태": ["완료", "대기", "완료"]})
display(status_sample)

,상태
0,완료
1,대기
2,완료


In [ ]:
# [실습] map()과 replace()
# 1. map()으 "상태"에서 "완료"를 "종료"로 바꾸세요 .
# 2. 같은 변경을 replace()로 수행하시오.
# 3. 매핑되지 않은 "대기"의 결과가 어떻게 다른지 확인하시오.


## **8) 숫자를 구간으로 나누기 `pd.cut()`**

`pd.cut()`은 숫자 데이터를 지정한 구간으로 나누고, 각 구간에 이름을 붙이는 함수이다.

예를 들어 매출을 '소·중·대', 점수를 '하·중·상'으로 분류할 때 사용한다.

### **기본 문법**

```python
pd.cut(데이터, bins=경계값, labels=구간이름)
```

### **[예제] 매출을 소·중·대로 분류하기**

```python
df["등급"] = pd.cut(
    df["매출"],
    bins=[0, 500000, 1000000, 10000000],
    labels=["소", "중", "대"]
)

display(df[["매출", "등급"]])
```

| 매출 범위 | 등급 |
|---|---|
| 0 초과 ~ 500,000 이하 | 소 |
| 500,000 초과 ~ 1,000,000 이하 | 중 |
| 1,000,000 초과 ~ 10,000,000 이하 | 대 |

### **주요 인자**

| 인자 | 설명 |
|---|---|
| `bins` | 구간의 경계값 지정. 경계값 4개 → 구간 3개 |
| `labels` | 각 구간의 이름 지정. 구간 개수와 같아야 한다 |
| `right=True` | 오른쪽 경계값 포함 (기본값) |
| `right=False` | 왼쪽 경계값 포함 |
| `include_lowest=True` | 첫 번째 구간에 가장 작은 경계값도 포함 |

### **경계값 포함 여부**

- `right=True` : `(0, 500000]` → 0 제외, 500000 포함
- `right=False` : `[0, 500000)` → 0 포함, 500000 제외
- `include_lowest=True` : `right=True`일 때 첫 번째 경계값인 0도 포함

### **주의사항**

- `bins=3`처럼 정수를 지정하면 데이터의 최솟값부터 최댓값까지 같은 폭의 구간 3개로 나눈다.
- 구간의 폭이 같아도 각 구간에 포함되는 데이터 개수는 다를 수 있다.
- 지정한 구간에 포함되지 않는 값은 `NaN`이 된다.
- 위 기본 예제에서 0 이하 또는 10,000,000 초과인 값은 `NaN`이 된다.
- 0도 첫 번째 구간에 포함하려면 `include_lowest=True`를 추가한다.


**공식 문서:** [pandas.cut()](https://pandas.pydata.org/docs/reference/api/pandas.cut.html)

In [ ]:
df = df_csv.copy()
# [예제] 매출을 세 구간으로 나누기
df["매출등급"] = pd.cut(df["매출"],
                     bins=[0, 500000, 1000000, 10000000],
                     labels=["소", "중", "대"])

display(df[["제품명", "매출", "매출등급"]].head())
print(df["매출등급"].value_counts())


,제품명,매출,매출등급
0,노트북,2400000,대
1,책상,900000,중
2,커피,300000,소
3,모니터,1200000,대
4,간식세트,480000,소


매출등급
중    6
대    6
소    3
Name: count, dtype: int64


In [ ]:
### **[실습] pd.cut(): 설비 온도를 구간으로 분류하기**
# 다음은 설비에서 측정한 온도 데이터이다.
temp_df = pd.DataFrame({"온도": [58.2, 72.5, 86.0, 91.3, 77.8, 65.0]})

# [온도 분류 기준]
# 0 초과 ~ 60 이하  → 저온
# 60 초과 ~ 85 이하 → 정상
# 85 초과 ~ 100 이하 → 고온

# 1. pd.cut()을 사용하여 온도를 세 구간으로 분류하시오.
#    bins=[0, 60, 85, 100]
#    labels=["저온", "정상", "고온"]
#    분류 결과를 temp_df의 "구간" 열에 저장하고 표를 출력하시오.
#    힌트: right=True가 기본값이므로 오른쪽 경계값을 포함한다.

# 2. "구간" 열에 value_counts()를 적용하여
#    온도 구간별 데이터 개수를 출력하시오.
#    예상 결과: 정상 3개, 고온 2개, 저온 1개

# 3. 마지막 경계값을 100에서 90으로 변경하여
#    bins=[0, 60, 85, 90]으로 다시 분류하시오.
#    labels는 기존과 동일하게 사용하고,
#    분류 결과를 limited 변수에 저장하시오.

# 4. limited에 isna().sum()을 적용하여
#    구간에 포함되지 않아 NaN이 된 값의 개수를 출력하시오.
#    예상 결과: 경계 밖 값: 1
#    힌트: 91.3은 마지막 경계값 90을 초과하므로 NaN이 된다.


## **9) 함수를 열에 적용하기 `apply()`**

`apply()`는 **데이터에 직접 만든 함수를 적용할 때** 사용한다.

예를 들어 매출이 100만 원 이상이면 "고액", 아니면 "일반"으로 구분하는 함수를 만들어 열 전체에 적용할 수 있다.

### **① 열의 값에 함수 적용하기**

```python
def grade(value):
    if value >= 1000000:
        return "고액"
    return "일반"

df["등급"] = df["매출"].apply(grade)
```

`df["매출"].apply(grade)`는 매출 열의 값을 하나씩 `grade()` 함수에 전달하고, 반환된 결과를 모아 새로운 "등급" 열에 저장한다.

**주의: 함수 이름 뒤에 괄호를 붙이지 않는다.**

```python
df["등급"] = df["매출"].apply(grade)    # 올바른 사용
# df["등급"] = df["매출"].apply(grade()) # 잘못된 사용
```

- `apply(grade)` : 적용할 함수를 전달한다.
- `apply(grade())` : 함수를 즉시 호출하려는 형태다. 위 함수는 필요한 인자가 없어 오류가 발생한다.

### **② lambda 함수 사용하기**

간단한 계산은 `lambda`로 작성할 수도 있다.

```python
df["매출_만원"] = df["매출"].apply(lambda x: x / 10000)
```

다만 위 계산은 `apply()` 없이도 가능하다.

```python
df["매출_만원"] = df["매출"] / 10000
```


### **③ 여러 열을 함께 사용하는 경우**

여러 열의 값을 함께 확인해야 한다면 `DataFrame.apply()`에 `axis=1`을 지정한다.

```python
def check_sales(row):
    if row["매출"] >= 1000000 and row["수량"] >= 10:
        return "우수"
    return "일반"

df["평가"] = df.apply(check_sales, axis=1)
```

- `axis=1` : 각 행을 하나씩 함수에 전달한다.
- `row["매출"]` : 현재 행의 매출 값
- `row["수량"]` : 현재 행의 수량 값

매출과 수량이라는 **두 열의 값을 함께 확인하여** 평가 결과를 만든다.

### **④ `apply()` 사용**

| 작업 | 우선 사용할 방법 |
|---|---|
| 단순 덧셈, 곱셈, 나눗셈 | `+`, `*`, `/` |
| 딕셔너리로 값 변경 | `map()` |
| 조건에 따라 값 분류 | `np.where()` |
| 직접 만든 함수 적용 | `apply()` |



공식 문서: [Series.apply()](https://pandas.pydata.org/docs/reference/api/pandas.Series.apply.html) · [DataFrame.apply()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)

In [ ]:
# [예제] 함수와 람다를 열에 적용하기
# 내용: 매출 금액에 따라 A·B·C 등급을 매기는 함수를 만들어 "등급" 열을 추가하고, 람다로 제품명 글자 수와 단가 열을 만든다.

# 1. 등급 규칙을 함수로 만든다. 값 하나(매출)를 받아 등급 글자 하나를 돌려준다.
#    200만 이상 → "A", 100만 이상 → "B", 나머지 → "C"
def sales_grade(value):
    if value >= 2000000:
        return "A"
    elif value >= 1000000:
        return "B"
    return "C"
df1 = df.copy()

df1


,주문번호,주문일,지역,제품군,제품명,수량,매출,매출등급
0,O001,2026-01-05,서울,전자,노트북,2,2400000,대
1,O002,2026-01-12,부산,가구,책상,3,900000,중
2,O003,2026-01-20,서울,식품,커피,10,300000,소
3,O004,2026-02-03,대전,전자,모니터,4,1200000,대
4,O005,2026-02-14,부산,식품,간식세트,8,480000,소
5,O006,2026-02-25,서울,가구,의자,5,750000,중
6,O007,2026-03-02,대전,전자,키보드,7,560000,중
7,O008,2026-03-11,부산,가구,수납장,2,800000,중
8,O009,2026-03-19,서울,전자,태블릿,3,2100000,대
9,O010,2026-03-28,대전,식품,차세트,6,420000,소


In [ ]:

# 2. apply() 에 함수 이름만 넘긴다 (괄호 없음). 매출 열의 값 15개가 차례로 함수에 들어가 등급 15개가 나온다.

df1["등급"] = df1["매출"].apply(sales_grade)
df1

,주문번호,주문일,지역,제품군,제품명,수량,매출,매출등급,등급
0,O001,2026-01-05,서울,전자,노트북,2,2400000,대,A
1,O002,2026-01-12,부산,가구,책상,3,900000,중,C
2,O003,2026-01-20,서울,식품,커피,10,300000,소,C
3,O004,2026-02-03,대전,전자,모니터,4,1200000,대,B
4,O005,2026-02-14,부산,식품,간식세트,8,480000,소,C
5,O006,2026-02-25,서울,가구,의자,5,750000,중,C
6,O007,2026-03-02,대전,전자,키보드,7,560000,중,C
7,O008,2026-03-11,부산,가구,수납장,2,800000,중,C
8,O009,2026-03-19,서울,전자,태블릿,3,2100000,대,A
9,O010,2026-03-28,대전,식품,차세트,6,420000,소,C


In [ ]:

# 3. 규칙이 한 줄이면 람다로 바로 적는다. name 자리에 제품명이 하나씩 들어간다.
df1["제품명_길이"] = df1["제품명"].apply(lambda name: len(name))
df1

,주문번호,주문일,지역,제품군,제품명,수량,매출,매출등급,등급,제품명_길이
0,O001,2026-01-05,서울,전자,노트북,2,2400000,대,A,3
1,O002,2026-01-12,부산,가구,책상,3,900000,중,C,2
2,O003,2026-01-20,서울,식품,커피,10,300000,소,C,2
3,O004,2026-02-03,대전,전자,모니터,4,1200000,대,B,3
4,O005,2026-02-14,부산,식품,간식세트,8,480000,소,C,4
5,O006,2026-02-25,서울,가구,의자,5,750000,중,C,2
6,O007,2026-03-02,대전,전자,키보드,7,560000,중,C,3
7,O008,2026-03-11,부산,가구,수납장,2,800000,중,C,3
8,O009,2026-03-19,서울,전자,태블릿,3,2100000,대,A,3
9,O010,2026-03-28,대전,식품,차세트,6,420000,소,C,3


In [ ]:

# 4. 행 전체(여러 열)를 보고 계산할 때는 df.apply(..., axis=1). row 에 행 하나가 들어오고 row["열이름"] 으로 값을 꺼낸다.
df1["단가_apply"] = df1.apply(lambda row: row["매출"] / row["수량"], axis=1)
df1

,주문번호,주문일,지역,제품군,제품명,수량,매출,매출등급,등급,제품명_길이,단가_apply
0,O001,2026-01-05,서울,전자,노트북,2,2400000,대,A,3,1200000.0
1,O002,2026-01-12,부산,가구,책상,3,900000,중,C,2,300000.0
2,O003,2026-01-20,서울,식품,커피,10,300000,소,C,2,30000.0
3,O004,2026-02-03,대전,전자,모니터,4,1200000,대,B,3,300000.0
4,O005,2026-02-14,부산,식품,간식세트,8,480000,소,C,4,60000.0
5,O006,2026-02-25,서울,가구,의자,5,750000,중,C,2,150000.0
6,O007,2026-03-02,대전,전자,키보드,7,560000,중,C,3,80000.0
7,O008,2026-03-11,부산,가구,수납장,2,800000,중,C,3,400000.0
8,O009,2026-03-19,서울,전자,태블릿,3,2100000,대,A,3,700000.0
9,O010,2026-03-28,대전,식품,차세트,6,420000,소,C,3,70000.0


In [ ]:

# 5. 만든 세 열을 확인하고, 등급별 건수를 센다.
display(df1[["제품명", "매출", "등급", "제품명_길이", "단가_apply"]].head())
print(df1["등급"].value_counts())   # A 3건, B 3건, C 9건

,제품명,매출,등급,제품명_길이,단가_apply
0,노트북,2400000,A,3,1200000.0
1,책상,900000,C,2,300000.0
2,커피,300000,C,2,30000.0
3,모니터,1200000,B,3,300000.0
4,간식세트,480000,C,4,60000.0


등급
C    9
A    3
B    3
Name: count, dtype: int64


In [ ]:
# [실습] apply()와 Pandas 기본 연산
# 문제: 제품별 매출 데이터를 이용하여 등급, 제품명 길이, 단가를 계산한다.

import pandas as pd

# 실습 데이터
df = pd.DataFrame({
    "제품명": ["노트북", "무선마우스", "키보드"],
    "매출": [1500000, 300000, 800000],
    "수량": [3, 10, 4]
})

display(df)


# 1. 매출에 따라 등급을 반환하는 grade(value) 함수를 만든다.
#    100만원 이상 → "고액"
#    100만원 미만 → "일반"



# 2. apply()를 사용하여 "등급" 열을 추가한다.



# 3. str.len()을 사용하여 제품명 글자 수를 계산하고
#    "제품명_길이" 열을 추가한다.



# 4. 매출을 수량으로 나누어 "단가" 열을 추가한다.



# 5. 제품명, 매출, 등급, 제품명_길이, 단가 열을 출력한다.
#    등급별 건수도 value_counts()로 출력한다.

,제품명,매출,수량
0,노트북,1500000,3
1,무선마우스,300000,10
2,키보드,800000,4


# **2. 표 합치기: merge()와 concat()**
## **1) `merge()`: 공통 열을 기준으로 표 합치기**

 주문 정보, 고객 정보처럼 데이터가 여러 표에 나뉘어 저장되는 경우가 많다.

`merge()`는 **두 표에서 공통으로 가지고 있는 열(키)을 기준으로 데이터를 옆으로 합칠 때** 사용한다.

예를 들어 주문 표에는 고객번호가 있고, 고객 표에는 고객번호와 고객 이름이 있다면 고객번호를 기준으로 두 표를 합칠 수 있다.

**기본 형식**

```python
pd.merge(왼쪽표, 오른쪽표, on="공통열", how="left")
```

- `on` : 두 표를 연결할 기준 열
- `how` : 어떤 데이터를 남길지 정하는 옵션

### **`how` 옵션 비교**

| `how` | 의미 |
|---|---|
| `"inner"` (기본값) | 양쪽 표에 공통으로 있는 키만 남긴다 |
| `"left"` | 왼쪽 표의 데이터를 유지하고 오른쪽 정보를 추가한다 |
| `"outer"` | 양쪽 표의 모든 키를 포함한다 |

`how="left"`는 주문 표의 데이터를 유지하면서 고객 이름처럼 추가 정보를 붙일 때 유용하다.

오른쪽 표에 일치하는 값이 없으면 해당 정보는 `NaN`으로 표시된다.

### **주의: 합친 후 행 개수 확인하기**

`left` 방식은 왼쪽 표의 행을 유지하지만, **오른쪽 표에 같은 키가 여러 개 있으면 결과 행 수가 늘어날 수 있다.**

따라서 표를 합친 뒤 행 개수를 확인하는 것이 좋다.

```python
result = pd.merge(왼쪽표, 오른쪽표, on="고객번호", how="left")

print(result.shape)     # 결과의 (행 개수, 열 개수)
print(result.shape[0])  # 결과의 행 개수
```

### **추가 옵션**

공통 열의 이름이 서로 다르면 `left_on`과 `right_on`을 사용한다.

```python
pd.merge(왼쪽표, 오른쪽표,
         left_on="고객번호", right_on="고객ID",
         how="left")
```

양쪽 표에 같은 이름의 열이 있으면 합친 결과에 `_x`, `_y`가 붙을 수 있다.

```python
pd.merge(왼쪽표, 오른쪽표,
         on="고객번호",
         suffixes=("_주문", "_고객"))
```

` suffixes`를 사용하면 겹치는 열 이름에 붙일 글자를 직접 지정할 수 있다.

**핵심:** `merge()`는 공통 열을 기준으로 두 표를 연결한다. `how` 옵션에 따라 결과에 남는 데이터가 달라진다.

공식 문서: [Pandas merge()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) · [표 합치기 안내](https://pandas.pydata.org/docs/user_guide/merging.html)

In [ ]:
# [예제] how="left" 와 how="inner" 비교
jobs = pd.DataFrame({"작업번호": ["W001", "W002", "W003", "W004"],
                     "라인": ["A라인", "B라인", "C라인", "A라인"],
                     "금액": [1800000, 650000, 240000, 1200000]})       # 작업 표 (왼쪽)


In [ ]:
jobs

,작업번호,라인,금액
0,W001,A라인,1800000
1,W002,B라인,650000
2,W003,C라인,240000
3,W004,A라인,1200000


In [ ]:
managers = pd.DataFrame({"라인": ["A라인", "B라인"],                    # 담당자 표 (오른쪽). C라인 담당자는 없다
                         "담당자": ["정하늘", "오세진"]})


In [ ]:
managers

,라인,담당자
0,A라인,정하늘
1,B라인,오세진


In [ ]:

left = pd.merge(jobs, managers, on="라인", how="left")     # 왼쪽(작업) 표의 행을 전부 유지

print("left :", left.shape)     # (4, 4) 4건 모두 남고 C라인 담당자는 NaN

display(left)



left : (4, 4)


,작업번호,라인,금액,담당자
0,W001,A라인,1800000,정하늘
1,W002,B라인,650000,오세진
2,W003,C라인,240000,NaN
3,W004,A라인,1200000,정하늘


In [ ]:
inner = pd.merge(jobs, managers, on="라인", how="inner")   # 양쪽에 다 있는 라인만

print("inner:", inner.shape)    # (3, 4) C라인 작업이 빠진다
display(inner)

inner: (3, 4)


,작업번호,라인,금액,담당자
0,W001,A라인,1800000,정하늘
1,W002,B라인,650000,오세진
2,W004,A라인,1200000,정하늘


In [ ]:
# 담당자 표에 금액 열 추가
managers2 = managers.copy()
managers2["금액"] = [100000, 200000]

inner = pd.merge(
    jobs,
    managers2,
    on="라인",
    how="inner",
    suffixes=("_left", "_right")
)

display(inner)


,작업번호,라인,금액_left,담당자,금액_right
0,W001,A라인,1800000,정하늘,100000
1,W002,B라인,650000,오세진,200000
2,W004,A라인,1200000,정하늘,100000


### **[실습] 이름이 다른 키로 병합하기**

In [ ]:
# [실습] 서로 다른 열 이름으로 merge()
# 1. managers의 "라인" 열 이름을 "담당라인"으로 바꾼 managers_alt를 만드시오.


# 2. jobs의 "라인"과 managers_alt의 "담당라인"을 기준으로
#    왼쪽 병합하여 담당자 정보를 붙이시오.


# 3. 병합 결과에서 담당자가 없는 행의 개수를 확인하시오.


## **실습 3: 반품 표시하기**

`superstore.xlsx`의 `Returns` 시트에는 반품된 주문번호(`Order ID`)와 `Returned`(`Yes`) 열만 있다. 주문 표 `orders`에 반품 여부를 붙여 반품으로 표시된 주문상품 행 수와 매출을 구한다.

1. `Returns` 시트를 `returns`에 불러오고 크기를 확인한다
2. `orders`와 `returns`를 `Order ID` 기준으로 `how="left"`로 합쳐 `orders_r`에 담고, 행 수가 그대로인지 확인한다
3. `Returned` 열의 빈 값(`NaN`) 개수를 센다. 반품 표에 없던 주문은 빈 값이 된다
4. `Returned`가 `"Yes"`인 행만 골라 반품으로 표시된 행 수와 반품 매출 합계를 출력한다


In [ ]:
# [실습] 반품 표시하기
# 1. store_path의 Returns 시트를 returns로 불러오시오.


# 2. orders와 returns를 Order ID로 왼쪽 병합하시오.


# 3. 병합 결과의 크기와 Returned 열의 결측값 개수를 출력하시오.


# 4. Returned가 Yes인 행의 개수와 Sales 합계를 출력하시오.


In [ ]:
orders.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


In [ ]:
returns.tail()

,Order ID,Returned
291,US-2026-136679,Yes
292,US-2026-147886,Yes
293,US-2026-147998,Yes
294,US-2026-151127,Yes
295,US-2026-155999,Yes


## **2) `concat()`: 키 없이 축을 따라 이어 붙이기**

`merge()`는 공통 열의 값을 맞춰 붙이고, `concat()`은 값을 보지 않고 그대로 이어 붙인다. 1월 작업 표와 3월 작업 표처럼 **열 구성이 같은 표를 위아래로** 합칠 때 가장 많이 쓴다.

```python
pd.concat([jan, mar], ignore_index=True)   # axis=0 (기본값): 위아래로. 행이 늘어난다
pd.concat([left, right], axis=1)           # axis=1: 옆으로. 같은 인덱스끼리 짝
```

| 옵션 | 뜻 | 예 |
|---|---|---|
| `axis` | `0`(기본) 위아래로, 열 이름 기준으로 맞춤 · `1` 옆으로, 행 인덱스 기준으로 맞춤 | `axis=1` |
| `ignore_index` | `True`면 기존 인덱스 대신 0부터 번호를 다시 매긴다. 위아래로 붙일 때 주로 쓴다 | `ignore_index=True` |
| `join` | `"outer"`(기본) 모두 유지, 빈 곳은 `NaN` · `"inner"` 공통 열(axis=0) 또는 공통 인덱스(axis=1)만 남긴다 | `join="inner"` |

위아래로 붙이면 각 표의 인덱스가 그대로 남아 `0, 1, 2, 0, 1, 2, 3`처럼 겹친다. 인덱스가 겹치면 `loc`로 고를 때 여러 행이 함께 나오므로 `ignore_index=True`로 번호를 다시 매긴다.

| 구분 | `merge()` | `concat()` |
|---|---|---|
| 기준 | 공통 열(키)의 값이 같은 행끼리 | 축(axis)을 따라 단순 연결 |
| 상황 | 작업 표에 담당자·검사 정보 덧붙이기 | 월별 파일, 라인별 파일을 하나로 합치기 |


공식 문서: [pandas.concat()](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) · [표 합치기 안내](https://pandas.pydata.org/docs/user_guide/merging.html)

In [ ]:
# [예제] concat() 으로 위아래·옆으로 붙이기
jan = pd.DataFrame({"작업번호": ["W001", "W002", "W003"],
                    "라인": ["A라인", "B라인", "A라인"],
                    "금액": [1800000, 650000, 240000]})                # 1월 작업
mar = pd.DataFrame({"작업번호": ["W007", "W008", "W009", "W010"],
                    "라인": ["C라인", "B라인", "A라인", "C라인"],
                    "금액": [450000, 980000, 1500000, 520000]})         # 3월 작업
print(pd.concat([jan, mar]))       # 인덱스가 겹친다: [0, 1, 2, 0, 1, 2, 3]

print(pd.concat([jan, mar]).index.tolist())       # 인덱스가 겹친다: [0, 1, 2, 0, 1, 2, 3]
both = pd.concat([jan, mar], ignore_index=True)   # 0부터 다시 번호를 매긴다
display(both)                                     # 3행 + 4행 = 7행


   작업번호   라인       금액
0  W001  A라인  1800000
1  W002  B라인   650000
2  W003  A라인   240000
0  W007  C라인   450000
1  W008  B라인   980000
2  W009  A라인  1500000
3  W010  C라인   520000
[0, 1, 2, 0, 1, 2, 3]


,작업번호,라인,금액
0,W001,A라인,1800000
1,W002,B라인,650000
2,W003,A라인,240000
3,W007,C라인,450000
4,W008,B라인,980000
5,W009,A라인,1500000
6,W010,C라인,520000


In [ ]:

left_part = pd.DataFrame({"작업번호": ["W001", "W002", "W003"], "품목명": ["DC모터", "온도센서", "볼밸브"]})
right_part = pd.DataFrame({"수량": [4, 6, 12], "금액": [1800000, 650000, 240000]})
display(pd.concat([left_part, right_part], axis=1))   # 같은 인덱스(0, 1, 2)끼리 옆으로

,작업번호,품목명,수량,금액
0,W001,DC모터,4,1800000
1,W002,온도센서,6,650000
2,W003,볼밸브,12,240000


In [ ]:
# [실습] concat()
# 문제: A라인과 B라인의 불량 기록을 하나의 표로 합치고,
#       전체 불량수 합계를 계산하시오.

# 실습 데이터
line_a = pd.DataFrame({"라인": ["A라인", "A라인"], "불량수": [3, 1]})
line_b = pd.DataFrame({"라인": ["B라인", "B라인", "B라인"], "불량수": [2, 4, 0]})

# 1. 두 표를 위아래로 합치고 인덱스를 0부터 다시 설정하시오.


# 2. 합친 표와 크기(shape)를 출력하시오.


# 3. 불량수의 전체 합계를 출력하시오.


# **3. 그룹별 집계와 표 모양 바꾸기**

## **1) `groupby()`**

"지역별 매출 평균", "설비별 불량률"처럼 **같은 값을 가진 데이터끼리 묶어 계산하는 것**을 그룹별 집계라고 한다. SQL의 `GROUP BY`와 비슷한 기능이다.

**기본 형식**

```python
df.groupby("묶을열")["계산할열"].계산()
```

예를 들어 지역별 평균 매출을 계산하려면 다음과 같이 작성한다.

```python
df.groupby("지역")["매출"].mean()
```

**실행 순서**

```text
1단계  groupby("지역")  → 지역이 같은 데이터끼리 묶기
2단계  ["매출"]         → 각 그룹에서 매출 열 선택
3단계  .mean()          → 지역별 평균 매출 계산
```

그룹별 집계에는 다음 메서드를 사용할 수 있다.

| 메서드 | 의미 |
|---|---|
| `sum()` | 그룹별 합계 |
| `mean()` | 그룹별 평균 |
| `count()` | 그룹별 결측값을 제외한 값의 개수 |
| `size()` | 그룹별 전체 행 개수 |
| `max()` | 그룹별 최댓값 |
| `min()` | 그룹별 최솟값 |

한 열에 하나의 집계 함수를 적용하면 일반적으로 결과가 Series로 반환되고, 그룹 기준 값이 인덱스가 된다.

```python
# 지역별 매출 합계 계산
# 그룹 기준인 "지역"이 인덱스가 된다. (기본값)
df.groupby("지역")["매출"].sum()

# 출력 예시 (실제 합계는 데이터에 따라 달라짐)
# 지역
# 대전    1200000
# 부산    2500000
# 서울    3800000
# Name: 매출, dtype: int64

# "지역"은 인덱스, "매출"은 Series의 이름이다.
# 결과는 DataFrame이 아닌 Series이다.

```
`reset_index()`를 사용하면 그룹 기준 값을 일반 열로 바꿀 수 있다.

```python
df.groupby("지역")["매출"].mean().reset_index()
```

### **여러 계산을 한 번에 하기 `agg()`**

`agg()`는 **하나의 그룹에 여러 집계 함수를 적용할 때** 사용한다.

```python
df.groupby("지역")["매출"].agg(["sum", "mean", "count"])
```

위 코드는 지역별 매출 합계, 평균, 개수를 한 번에 계산한다. 결과는 DataFrame이다.

묶는 기준이 여러 개라면 리스트로 지정한다.

```python
df.groupby(["지역", "제품군"])["매출"].sum()
```

### **`agg()`에 lambda 함수 사용하기**

기본 집계 메서드에 없는 계산을 직접 작성하려면 `lambda`를 사용할 수 있다.

예를 들어 지역별 매출의 최댓값과 최솟값의 차이를 계산한다.

```python
df.groupby("지역")["매출"].agg(
    lambda x: x.max() - x.min()
)
```

여기서 `x`는 매출 값 하나가 아니라 **각 지역에 속한 매출 값들을 담은 Series**이다.

앞에서 배운 `Series.apply()`는 값을 하나씩 함수에 전달할 수 있지만, 이 예제의 `groupby().agg()`는 그룹의 값들을 묶어서 함수에 전달한다.

단순한 합계와 평균 등은 `sum()`, `mean()` 같은 기본 집계 메서드를 우선 사용한다.

### **가장 큰 값의 인덱스 찾기 `idxmax()`**

`max()`는 가장 큰 **값**을 반환하고, `idxmax()`는 가장 큰 값이 위치한 **인덱스 레이블**을 반환한다.

```python
region_mean = df.groupby("지역")["매출"].mean()

print(region_mean.max())     # 지역별 평균 중 가장 큰 값
print(region_mean.idxmax())  # 평균이 가장 높은 지역 이름
```

`groupby()` 결과에서 지역명이 인덱스로 사용되므로 `idxmax()`로 가장 높은 평균 매출을 가진 지역을 찾을 수 있다.

가장 작은 값의 인덱스를 찾으려면 `idxmin()`을 사용한다.

- `max()` : 최댓값
- `idxmax()` : 최댓값의 인덱스 레이블
- `min()` : 최솟값
- `idxmin()` : 최솟값의 인덱스 레이블

참고로 최댓값이 여러 개라면 `idxmax()`는 첫 번째 최댓값의 인덱스 레이블을 반환한다.

NumPy의 `np.argmax()`는 위치 번호를 반환하지만, Pandas의 `idxmax()`는 인덱스 레이블을 반환한다.


공식 문서: [Pandas groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · [그룹 연산 안내](https://pandas.pydata.org/docs/user_guide/groupby.html) · [Series.idxmax()](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html)

In [ ]:
import pandas as pd

# [예제] groupby()로 지역별 매출 집계하기

# 1. 실습 데이터 만들기
df = pd.DataFrame({
    "지역": ["서울", "서울", "서울", "부산", "부산", "부산"],
    "제품군": ["노트북", "마우스", "노트북", "노트북", "마우스", "노트북"],
    "매출": [100, 200, 300, 100, 250, 400]
})

display(df)



,지역,제품군,매출
0,서울,노트북,100
1,서울,마우스,200
2,서울,노트북,300
3,부산,노트북,100
4,부산,마우스,250
5,부산,노트북,400


In [ ]:

# 2. 지역별 평균 매출 계산
#    지역이 같은 데이터끼리 묶어서 평균을 구한다.
#    서울: (100 + 200 + 300) / 3 = 200
#    부산: (100 + 250 + 400) / 3 = 250

print("지역별 평균 매출:")
print(df.groupby("지역")["매출"].mean())



지역별 평균 매출:
지역
부산    250.0
서울    200.0
Name: 매출, dtype: float64


In [ ]:


# 3. agg(): 지역별 매출의  합계, 평균, 개수를 한 번에 계산
print("지역별 합계, 평균, 개수:")
print(df.groupby("지역")["매출"].agg(["sum", "mean", "count"]))
print("-" * 40)


지역별 합계, 평균, 개수:
    sum   mean  count
지역                   
부산  750  250.0      3
서울  600  200.0      3
----------------------------------------


In [ ]:


# 4. 묶는 기준을 2개로 설정
#    지역과 제품군이 모두 같은 데이터끼리 묶어 합계를 구한다.
print("지역 및 제품군별 매출 합계:")
print(df.groupby(["지역", "제품군"])["매출"].sum())



지역 및 제품군별 매출 합계:
지역  제품군
부산  노트북    500
    마우스    250
서울  노트북    400
    마우스    200
Name: 매출, dtype: int64


In [ ]:

# 5. agg()와 lambda로 지역별 매출 범위 계산
#    각 지역의 최댓값에서 최솟값을 뺀다.
#    서울: 300 - 100 = 200
#    부산: 400 - 100 = 300

print("지역별 매출 범위:")
print(df.groupby("지역")["매출"].agg(lambda x: x.max() - x.min()))

지역별 매출 범위:
지역
부산    300
서울    200
Name: 매출, dtype: int64


In [ ]:
import pandas as pd

# [실습] groupby()
# 문제: 제품군별 매출과 수량을 집계하고,
#       매출 평균이 가장 높은 제품군을 찾는다.

# 실습 데이터
df = pd.DataFrame({
    "제품군": ["전자", "가구", "전자", "식품", "가구", "전자"],
    "매출": [300, 150, 200, 80, 250, 400],
    "수량": [2, 3, 1, 4, 2, 5]
})

display(df)


# 1. 제품군별 매출 합계를 구하고 큰 순서대로 정렬한다.
# 출력: 전자 900, 가구 400, 식품 80



# 2. 제품군별 수량의 합계, 평균, 건수를 한 번에 계산한다.



# 3. 매출 평균이 가장 높은 제품군을 idxmax()로 찾는다.
# 출력: 매출 평균 1위 제품군: 전자

,제품군,매출,수량
0,전자,300,2
1,가구,150,3
2,전자,200,1
3,식품,80,4
4,가구,250,2
5,전자,400,5


### **참고: `groupby()` 추가 옵션과 `agg()` 활용**

#### **① `as_index=False`: 그룹 기준을 일반 열로 유지**

`groupby()`는 기본적으로 묶은 열을 인덱스로 사용한다.

`as_index=False`를 지정하면 **묶은 열을 일반 열로 유지**할 수 있다.

```python
# 지역이 인덱스가 된다. (기본값)
df.groupby("지역")["매출"].sum()

# 지역이 일반 열로 남는다.
df.groupby("지역", as_index=False)["매출"].sum()
```

#### **② `dropna=False`: 결측값도 그룹에 포함**

기본적으로 그룹 기준 열의 결측값은 집계에서 제외된다.

`dropna=False`를 지정하면 **결측값도 하나의 그룹으로 묶어 계산**한다.

```python
df.groupby("지역", dropna=False)["매출"].sum()
```

#### **③ `agg()`: 열마다 다른 계산 적용**

`agg()`에 딕셔너리를 전달하면 **열마다 서로 다른 집계 함수를 적용**할 수 있다.

```python
# 매출은 합계, 수량은 평균
df.groupby("지역").agg({
    "매출": "sum",
    "수량": "mean"
})
```

| 기능 | 의미 |
|---|---|
| `as_index=False` | 그룹 기준을 일반 열로 유지 |
| `dropna=False` | 그룹 기준의 결측값도 집계에 포함 |
| `agg({열: 계산})` | 열마다 다른 집계 함수 적용 |

공식 문서: [Pandas groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · [DataFrameGroupBy.agg()](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)

In [ ]:
# [예제] 표 형태의 집계 결과와 열마다 다른 계산
# 앞의 실습에서 df가 다른 표로 바뀌었으므로 원래 매출 데이터로 복원한다.
df = df_csv.copy()
# 지역이 인덱스가 된다. (기본값)
df.groupby("지역")["매출"].sum()


,매출
지역,
대전,2730000
부산,4630000
서울,9450000


In [ ]:
# 방법 1. 집계 후 인덱스를 일반 열로 변환
display(df.groupby("지역")["매출"].sum().reset_index())

# 방법 2. 그룹 기준을 처음부터 일반 열로 유지
display(df.groupby("지역", as_index=False)["매출"].sum())

# 두 방법 모두 지역이 일반 열인 DataFrame을 반환

summary = df.groupby("제품군").agg({"매출": "sum", "수량": "mean"})   # 매출은 합계, 수량은 평균
display(summary)

,지역,매출
0,대전,2730000
1,부산,4630000
2,서울,9450000


,지역,매출
0,대전,2730000
1,부산,4630000
2,서울,9450000


,매출,수량
제품군,,
가구,4250000,2.600000
식품,1750000,7.250000
전자,10810000,4.833333


### **[실습] groupby 집계 결과**

In [ ]:
# [실습] groupby(): 그룹별 집계와 결과 형태 변경

# 실습 데이터 준비 (실행 후 문제를 푸시오)
group_sample = pd.DataFrame({
    "제품군": ["전자", "전자", "가구", "가구"],
    "매출": [100, 300, 200, 400],
    "수량": [2, 4, 1, 3]
})
display(group_sample)

# 1. 제품군별 매출 합계를 구하시오.
#    groupby()와 sum()을 사용한다.
#    결과에서 "제품군"이 인덱스가 되는지 확인하시오.

# 2. 제품군별 매출 합계를 계산하고,
#    reset_index()를 사용하여 "제품군"을 일반 열로 변환하시오.

# 3. as_index=False를 사용하여 제품군별 매출 합계를 구하시오.
#    2번과 3번의 결과가 같은지 비교하시오.

# 4. agg()를 사용하여 제품군별로 다음 항목을 한 번에 계산하시오.
#    - 매출: 합계(sum)
#    - 수량: 평균(mean)
#    힌트: agg({"매출": "sum", "수량": "mean"})
#    결과를 summary 변수에 저장하고 출력하시오.

,제품군,매출
0,전자,100
1,전자,300
2,가구,200
3,가구,400


### **참고: `transform()` — 원래 행 수를 유지하는 그룹 계산**

`groupby().mean()`은 그룹마다 한 줄로 줄어든다. `transform("mean")`은 그룹별 평균을 구한 뒤 **원래 행 자리마다** 다시 돌려준다. 결과의 길이가 원래 표와 같아서 새 열로 바로 붙일 수 있다.

```text
 지역   매출          groupby("지역")["매출"].mean()     groupby("지역")["매출"].transform("mean")
 서울  2400000        대전    682500.0                  1575000.0   (서울 평균)
 부산   900000        부산    926000.0                   926000.0   (부산 평균)
 서울   300000        서울   1575000.0                  1575000.0   (서울 평균)
 ...                  → 지역 3줄                         → 원래 15줄 그대로
```

자주 쓰는 곳은 두 가지다.

- 그룹 평균과 비교하기: `df["매출"] - df.groupby("지역")["매출"].transform("mean")`
- 같은 그룹의 평균으로 결측값 채우기: `df["온도"].fillna(df.groupby("라인")["온도"].transform("mean"))`



공식 문서: [transform()](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html)

In [ ]:
# [참고 예제] transform(): 그룹 평균을 원래 행마다 붙이기
# 원본 데이터를 복사하여 사용
df = df_csv.copy()
compare = df[["주문번호", "지역", "매출"]].copy()
compare["지역평균"] = compare.groupby("지역")["매출"].transform("mean")   # 행 수 15개 그대로
compare["평균대비"] = compare["매출"] - compare["지역평균"]                # 지역 평균보다 얼마나 큰지
display(compare.head())


,주문번호,지역,매출,지역평균,평균대비
0,O001,서울,2400000,1575000.0,825000.0
1,O002,부산,900000,926000.0,-26000.0
2,O003,서울,300000,1575000.0,-1275000.0
3,O004,대전,1200000,682500.0,517500.0
4,O005,부산,480000,926000.0,-446000.0


In [ ]:

# 같은 라인의 평균으로 결측값 채우기
temp = pd.DataFrame({"라인": ["A라인", "A라인", "B라인", "B라인", "B라인"],
                     "온도": [70.0, np.nan, 80.0, 84.0, np.nan]})
temp["온도"] = temp["온도"].fillna(temp.groupby("라인")["온도"].transform("mean"))   # A라인 빈 칸 70.0, B라인 빈 칸 82.0
print(temp)

    라인    온도
0  A라인  70.0
1  A라인  70.0
2  B라인  80.0
3  B라인  84.0
4  B라인  82.0


### **[참고 실습] 그룹별 평균 붙이기 (`transform()`)**

In [ ]:
# 실습 데이터 준비 (실행 후 문제를 푸시오)
transform_sample = pd.DataFrame({"지역": ["서울", "서울", "부산", "부산"], "매출": [100, 300, 200, 400]})
display(transform_sample)

,지역,매출
0,서울,100
1,서울,300
2,부산,200
3,부산,400


In [ ]:
# [참고 실습] transform() 기초
# 1. 지역별 평균 매출을 구해 "지역평균" 열에 넣으시오.
# 2. 지역평균과 각 행의 매출을 함께 출력하시오.


## **2) `pivot_table()`**

`pivot_table()`은 **데이터를 행과 열로 나누어 합계, 평균 등을 계산하는 기능**이다.

예를 들어 지역별·제품군별 매출을 가로세로 표로 정리할 수 있다.

Excel 피벗 테이블에서 행, 열, 값을 지정하는 것과 비슷하다.

**기본 형식**

```python
df.pivot_table(
    index="지역",
    columns="제품군",
    values="매출",
    aggfunc="sum"
)
```

| 인자 | 의미 |
|---|---|
| `index` | 행에 표시할 기준 |
| `columns` | 열에 표시할 기준 |
| `values` | 계산할 데이터 |
| `aggfunc` | 계산 방법 (`"sum"`, `"mean"` 등) |
| `fill_value=0` | 결과에서 비어 있는 값을 0으로 채우기 |
| `margins=True` | 행과 열에 전체 집계 결과(`All`) 추가 |

### **주요 옵션**

**① `aggfunc` : 계산 방법 지정**

`aggfunc`를 생략하면 기본적으로 평균(`mean`)을 계산한다.

매출 합계를 구하려면 `"sum"`을 지정한다.

```python
# 매출 합계
df.pivot_table(
    index="지역",
    columns="제품군",
    values="매출",
    aggfunc="sum"
)
```

**② `fill_value=0` : 비어 있는 결과 채우기**

특정 지역에서 판매하지 않은 제품군처럼 해당 조합의 데이터가 없으면 결과가 `NaN`으로 표시될 수 있다.

`fill_value=0`을 지정하면 이러한 빈 결과를 0으로 표시한다.

```python
df.pivot_table(
    index="지역",
    columns="제품군",
    values="매출",
    aggfunc="sum",
    fill_value=0
)
```

**③ `margins=True` : 전체 집계 추가**

행과 열에 `All`이라는 전체 집계 결과를 추가한다.

```python
df.pivot_table(
    index="지역",
    columns="제품군",
    values="매출",
    aggfunc="sum",
    margins=True
)
```

- `aggfunc="sum"` : 전체 합계 추가
- `aggfunc="mean"` : 전체 평균 추가

### **여러 계산을 한 번에 수행하기**

`aggfunc`에 여러 집계 함수를 리스트로 전달할 수 있다.

```python
df.pivot_table(
    index="지역",
    columns="제품군",
    values="매출",
    aggfunc=["sum", "count"]
)
```

이렇게 하면 매출 합계와 데이터 개수를 함께 계산한다.

결과의 열은 `sum`, `count`를 상위 구분으로 하고, 그 아래에 제품군이 표시되는 다중 인덱스 형태가 된다.

### **참고: `groupby()`와 `unstack()`**

두 개의 기준으로 묶은 `groupby()` 결과에 `unstack()`을 사용해도 비슷한 가로세로 표를 만들 수 있다.

```python
df.groupby(["지역", "제품군"])["매출"].sum().unstack()
```

`unstack()`은 기본적으로 마지막 인덱스 단계인 제품군을 열로 이동시킨다.

**핵심 정리**

- `groupby()` : 같은 값끼리 묶어 집계
- `pivot_table()` : 집계 결과를 행과 열로 펼쳐 표시
- `unstack()` : 인덱스의 한 단계를 열로 이동

공식 문서: [Pandas pivot_table()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) · [Pandas unstack()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html)

In [ ]:
# [예제] 지역 × 제품군 매출 표
display(df.pivot_table(index="지역", columns="제품군", values="매출"))

# [예제] 지역 × 제품군 매출 합계 표
display(df.pivot_table(index="지역", columns="제품군", values="매출", aggfunc="sum"))


# 빈 칸은 0 으로, 합계도 함께
display(df.pivot_table(index="지역", columns="제품군", values="매출", aggfunc="sum",
                       fill_value=0, margins=True))


제품군,가구,식품,전자
지역,,,
대전,NaN,970000.0,1760000.0
부산,2800000.0,480000.0,1350000.0
서울,1450000.0,300000.0,7700000.0


제품군,가구,식품,전자,All
지역,,,,
대전,0,970000,1760000,2730000
부산,2800000,480000,1350000,4630000
서울,1450000,300000,7700000,9450000
All,4250000,1750000,10810000,16810000


In [ ]:
import pandas as pd

# [예제] pivot_table()로 평균과 합계 계산하기

# 1. 설비 라인별 품목 금액 데이터 만들기
sample = pd.DataFrame({
    "라인": ["A라인", "A라인", "B라인", "B라인", "A라인", "B라인"],
    "품목": ["모터", "센서", "모터", "센서", "모터", "센서"],
    "금액": [180, 40, 90, 60, 150, 30]
})

display(sample)


# 2. pivot_table(): 라인별·품목별 평균 금액 계산
#    aggfunc를 생략하면 기본값은 mean(평균)이다.
#    A라인 모터 평균: (180 + 150) / 2 = 165

print("라인별·품목별 평균:")
display(sample.pivot_table(
    index="라인",
    columns="품목",
    values="금액"
))


# 3. aggfunc="sum": 라인별·품목별 합계 계산
#    A라인 모터 합계: 180 + 150 = 330

print("라인별·품목별 합계:")
display(sample.pivot_table(
    index="라인",
    columns="품목",
    values="금액",
    aggfunc="sum"
))


# 4. groupby()와 unstack(): 같은 합계 결과 확인
#    라인과 품목으로 묶어 합계를 구한 뒤,
#    품목을 열로 펼친다.

print("groupby()와 unstack() 결과:")
display(sample.groupby(["라인", "품목"])["금액"].sum().unstack())

,라인,품목,금액
0,A라인,모터,180
1,A라인,센서,40
2,B라인,모터,90
3,B라인,센서,60
4,A라인,모터,150
5,B라인,센서,30


라인별·품목별 평균:


품목,모터,센서
라인,,
A라인,165.0,40.0
B라인,90.0,45.0


라인별·품목별 합계:


품목,모터,센서
라인,,
A라인,330,40
B라인,90,90


groupby()와 unstack() 결과:


품목,모터,센서
라인,,
A라인,330,40
B라인,90,90


### **[실습] 피벗 테이블 기본**

In [ ]:
# 실습 데이터 준비
pivot_sample = pd.DataFrame({"라인": ["A", "A", "B"], "품목": ["모터", "센서", "모터"], "금액": [100, 30, 80]})
display(pivot_sample)

,라인,품목,금액
0,A,모터,100
1,A,센서,30
2,B,모터,80


In [ ]:
# [실습] pivot_table() 기초
# 1. pivot_sample의 라인을 행, 품목을 열로 놓으시오.
# 2. 값은 금액, 계산은 합계로 지정하시오.
# 3. 비어 있는 결과는 0으로 표시하시오.


## **실습 4: superstore 집계**

`orders`로 다음을 구한다.

1. 지역(`Region`)별 매출(`Sales`) 합계를 큰 순서로, 정수로 출력한다
2. 대분류(`Category`)별 매출 합계·평균·건수를 표로 표시한다
3. 주문일의 연도별 매출 합계를 출력한다 (`.dt.year`로 묶는다)
4. 지역 × 대분류 매출 합계 피벗 표를 만들고 정수로 표시한다
5. 배송 방식(`Ship Mode`)별 평균 이익(`Profit`)이 가장 높은 방식을 출력한다


In [ ]:
# 1. 지역별 매출 합계 (큰 순서, 정수)

# 2. 대분류별 합계·평균·건수

# 3. 연도별 매출 합계: 묶는 기준으로 열 이름 대신 Series(연도)를 넘길 수 있다

# 4. 지역 × 대분류 피벗 표

# 5. 배송 방식별 평균 이익이 가장 높은 방식


## **3) 가로 표를 세로로 바꾸기 `melt()`**

`melt()`는 **여러 열에 나누어 저장된 데이터를 세로로 모을 때** 사용한다.

예를 들어 월별 생산량이 `1월`, `2월`, `3월` 열로 나뉘어 있다면, 이를 `"설비"`, `"월"`, `"생산량"` 세 열로 정리할 수 있다.

**변경 전 (가로 표)**

| 설비 | 1월 | 2월 | 3월 |
|---|---:|---:|---:|
| M01 | 210 | 230 | 220 |
| M02 | 150 | 160 | 170 |

**변경 후 (세로 표)**

| 설비 | 월 | 생산량 |
|---|---|---:|
| M01 | 1월 | 210 |
| M02 | 1월 | 150 |
| M01 | 2월 | 230 |
| M02 | 2월 | 160 |
| M01 | 3월 | 220 |
| M02 | 3월 | 170 |

### **기본 사용법**

```python
pd.melt(
    df,
    id_vars="설비",
    var_name="월",
    value_name="생산량"
)
```

| 옵션 | 의미 |
|---|---|
| `id_vars` | 변경하지 않고 그대로 유지할 열 |
| `value_vars` | 세로로 모을 열 (생략하면 나머지 열 전체) |
| `var_name` | 기존 열 이름을 저장할 새 열 이름 |
| `value_name` | 기존 데이터 값을 저장할 새 열 이름 |

**실행 예제**

```python
import pandas as pd

# 1. 월별 생산량 데이터
df = pd.DataFrame({
    "설비": ["M01", "M02"],
    "1월": [210, 150],
    "2월": [230, 160],
    "3월": [220, 170]
})

display(df)

# 2. 가로 표를 세로 표로 변경
result = pd.melt(
    df,
    id_vars="설비",
    var_name="월",
    value_name="생산량"
)

display(result)

# 3. 결과 크기 확인
print(result.shape)  # (6, 3)
```

### **`pivot_table()`과 `melt()`의 차이**

| 기능 | 변경 방향 | 설명 |
|---|---|---|
| `pivot_table()` | 세로 → 가로 | 행과 열로 펼쳐 집계 |
| `melt()` | 가로 → 세로 | 여러 열의 데이터를 하나의 열로 모으기 |

`melt()`는 데이터를 더하거나 평균 내지 않고 **표의 모양만 변경한다.**

위 예제는 설비 2개와 월 3개이므로 결과는 **2 × 3 = 6행**이 된다.

**핵심:** `pivot_table()`은 데이터를 행과 열로 펼쳐 집계하고, `melt()`는 여러 열에 있는 데이터를 세로로 모은다. 두 기능은 모양을 바꾸는 방향이 반대이지만, 집계하는 `pivot_table()`과 집계하지 않는 `melt()`가 완전히 같은 작업을 거꾸로 수행하는 것은 아니다.

공식 문서: [Pandas melt()](https://pandas.pydata.org/docs/reference/api/pandas.melt.html) · [데이터 모양 바꾸기](https://pandas.pydata.org/docs/user_guide/reshaping.html)

In [ ]:
# [예제] melt() 로 월별 열을 세로로 모으기
wide = pd.DataFrame({"설비": ["M01", "M02"],
                     "1월": [210, 150], "2월": [230, 160], "3월": [220, 170]})
display(wide)

long = pd.melt(wide, id_vars="설비", var_name="월", value_name="생산량")   # 설비는 그대로, 월 열들을 세로로
display(long)
print(wide.shape, "->", long.shape)                                        # (2, 4) -> (6, 3)

print(long.groupby("월")["생산량"].sum())                                  # 세로 표라서 월별 합계를 바로 구한다
display(long.pivot_table(index="설비", columns="월", values="생산량", aggfunc="sum"))   # 다시 펼치면 원래 모양

,설비,1월,2월,3월
0,M01,210,230,220
1,M02,150,160,170


,설비,월,생산량
0,M01,1월,210
1,M02,1월,150
2,M01,2월,230
3,M02,2월,160
4,M01,3월,220
5,M02,3월,170


(2, 4) → (6, 3)
월
1월    360
2월    390
3월    390
Name: 생산량, dtype: int64


월,1월,2월,3월
설비,,,
M01,210,230,220
M02,150,160,170


In [ ]:
```python
# [실습] melt(): 가로 표를 세로 표로 변환하기
# 다음은 생산 라인별 분기 불량 수 데이터이다.
# melt()를 사용하여 가로 형태의 데이터를 세로 형태로 변환하고,
# 분기별 불량 수 합계와 불량이 가장 많은 라인·분기를 확인하시오.

quarter = pd.DataFrame({
    "라인": ["A라인", "B라인", "C라인"],
    "1분기": [12, 8, 15],
    "2분기": [10, 11, 9],
    "3분기": [7, 13, 6]
})
display(quarter)

# 1. melt()를 사용하여 가로 형태의 표를 세로 형태로 변환하시오.
#    - "라인" 열은 그대로 유지한다. (id_vars)
#    - "1분기", "2분기", "3분기"를 "분기" 열로 모은다. (var_name)
#    - 각 분기의 불량 수를 "불량수" 열에 저장한다. (value_name)
#    - 변환 결과를 long_q 변수에 저장하고 출력하시오.
#    - shape로 변환 후 행과 열의 개수를 확인하시오.
#    예상 결과: (9, 3)

# 2. long_q에서 분기별 불량 수 합계를 구하시오.
#    groupby("분기")와 sum()을 사용한다.
#    예상 결과: 1분기 35, 2분기 30, 3분기 26

# 3. 불량 수가 가장 많은 라인과 분기를 조회하시오.
#    long_q에서 nlargest(1, "불량수")를 사용한다.
#    예상 결과: C라인, 1분기, 불량수 15

# 4.  "1분기"와 "2분기"만 세로로 변환하시오.
#    - "라인" 열은 그대로 유지한다. (id_vars)
#    - value_vars=["1분기", "2분기"]를 지정한다.
#    - 열 이름은 "분기", "불량수"로 설정한다.
#    - 변환 결과를 long_two 변수에 저장하시오.
#    - len()으로 변환 후 전체 행 개수를 출력하시오.
#    예상 결과: 6
```

## **4. 미니 실습: 매출 데이터 조건별 조회 및 요약**

지금까지 학습한 `isin()`, 조건 필터링, `sort_values()`, `pivot_table()`을 활용하여 매출 데이터를 조회하고 요약하시오.

```python
# [실습] 매출 데이터 조건별 조회 및 요약

# 1. 원본 데이터 df_csv를 복사하여 df에 저장하시오.
#    "주문일" 열을 날짜형(datetime)으로 변환하시오.

# 2. 다음 조회 조건을 변수에 저장하시오.
#    - 조회 지역: 서울, 부산 → selected_regions
#    - 조회 제품군: 전자, 가구 → selected_categories
#    - 최소 매출: 500000 → minimum_sales
#    - 시작일: 2026-01-01 → start_date
#    - 종료일: 2026-05-31 → end_date

# 3. 다음 조건을 모두 만족하는 데이터를 조회하시오.
#    - 지역이 서울 또는 부산인 행 (isin 사용)
#    - 제품군이 전자 또는 가구인 행 (isin 사용)
#    - 매출이 500000 이상인 행
#    - 주문일이 2026-01-01부터 2026-05-31까지인 행
#    - 날짜에 시간이 포함되어 있어도 종료일 전체를 포함할 것
#    조회 결과를 매출 내림차순으로 정렬하여 result에 저장하시오.
#    힌트: isin(), &, >=, <, sort_values()

# 4. 조회된 데이터의 행 개수와 결과를 출력하시오.
#    힌트: len(), display()

# 5. result를 사용하여 지역별·제품군별 매출 합계표를 만드시오.
#    - 행(index): 지역
#    - 열(columns): 제품군
#    - 값(values): 매출
#    - 집계 함수(aggfunc): 합계(sum)
#    - 데이터가 없는 조합은 0으로 표시
#    결과를 summary 변수에 저장하고 출력하시오.
#    힌트: pivot_table(), fill_value=0
```

# **7. 자주 발생하는 오류**

## **merge 후 행 수가 늘어남**

오른쪽 표에 같은 키가 여러 번 있으면 왼쪽 행 하나에 여러 행이 붙어 행 수가 늘어난다. 합치기 전후로 `shape`을 비교하고, 오른쪽 표의 키가 겹치지 않는지 확인한다.

```python
print(df.shape, merged.shape)                 # 행 수가 그대로인지
print(managers["라인"].duplicated().sum())  # 오른쪽 키의 중복 여부 확인
```

## **merge 키 오류 (KeyError)**

```python
pd.merge(df, managers, on="라인")   # KeyError: '라인'
```

`on`에 적은 열이 두 표 중 하나에 없을 때 발생한다. 두 표의 키 열 이름이 서로 다르면 `left_on`, `right_on`으로 각각 지정한다.

## **concat 후 인덱스가 겹침**

위아래로 붙이면 각 표의 인덱스가 그대로 남아 `0, 1, 2, 0, 1`처럼 겹친다. `ignore_index=True`를 쓴다.

## **groupby 열 이름 오류 (KeyError)**

```python
df.groupby("지역")["매출 "].sum()   # KeyError: 'Column not found: 매출 '
```

열 이름에 공백 같은 오타가 있을 때 발생한다. `df.columns`로 정확한 이름을 확인한다.

## **pivot_table 결과가 합계가 아니라 평균**

`aggfunc`를 생략하면 평균(`"mean"`)이다. 합계가 필요하면 `aggfunc="sum"`을 적는다.

## **pd.cut 결과에 NaN**

값이 `bins` 범위 밖(가장 작은 경계 이하, 가장 큰 경계 초과)이면 `NaN`이 된다. `min()`, `max()`로 범위를 먼저 확인하고, 가장 작은 경계값도 넣으려면 `include_lowest=True`를 쓴다.

## **map 결과에 NaN**

딕셔너리에 없는 값은 `NaN`이 된다. `isna().sum()`으로 확인하고, 일부 값만 바꾸려면 `replace()`를 쓴다.

## **apply()에 괄호를 붙인 경우 (TypeError)**

```python
df["매출"].apply(sales_grade())   # TypeError: sales_grade() missing 1 required positional argument: 'value'
```

`apply()`에는 함수 이름만 넘긴다: `apply(sales_grade)`.


# **6. 확인문제**

아래 질문에 답할 수 있는지 확인한다.

1. `df.sort_values(["지역", "매출"], ascending=[True, False])`는 어떻게 정렬하는가?
2. `rank(method="min")`과 `rank(method="dense")`의 차이는 무엇인가?
3. `cumsum()`을 쓰기 전에 정렬하는 이유는 무엇인가?
4. `map()`과 `replace()`의 차이는 무엇인가?
5. `merge()`에서 `how="left"`와 `how="inner"`의 차이는 무엇인가?
6. `merge()`와 `concat()`은 각각 언제 쓰는가?
7. `df.groupby("지역")["매출"].mean()`은 어떤 순서로 계산되는가?
8. `pivot_table()`에서 `aggfunc`를 생략하면 어떻게 계산되는가?
9. `melt()`와 `pivot_table()`은 어떤 관계인가?


## **확인문제 정답**

### **1. `df.sort_values(["지역", "매출"], ascending=[True, False])`는 어떻게 정렬하는가?**

앞에 적은 `지역`으로 먼저 오름차순(가나다순) 정렬하고, 지역이 같은 행끼리는 `매출`로 내림차순(큰 값부터) 정렬한다. `ascending`도 열 순서대로 리스트로 적는다.

---

### **2. `rank(method="min")`과 `rank(method="dense")`의 차이는 무엇인가?**

둘 다 같은 값에는 같은 순위를 준다. 차이는 다음 순위다.

```text
 값      90  80  80  70
 min      1   2   2   4    같은 순위 수만큼 다음 순위를 건너뛴다
 dense    1   2   2   3    건너뛰지 않는다
```

---

### **3. `cumsum()`을 쓰기 전에 정렬하는 이유는 무엇인가?**

`cumsum()`은 현재 행 순서대로 차례로 더한다. 날짜순 누적처럼 순서가 의미 있는 누적 합을 구하려면 먼저 그 기준으로 정렬해야 한다. 마지막 누적값은 정렬과 상관없이 전체 합계와 같다.

---

### **4. `map()`과 `replace()`의 차이는 무엇인가?**

| 메서드 | 딕셔너리에 없는 값 | 쓰는 상황 |
|---|---|---|
| `map()` | `NaN`이 된다 | 모든 값을 새 이름으로 1:1 대응할 때 |
| `replace()` | 원래 값 그대로 | 일부 값만 고칠 때 |

---

### **5. `merge()`에서 `how="left"`와 `how="inner"`의 차이는 무엇인가?**

| `how` | 남는 행 | 상대 표에 키가 없을 때 |
|---|---|---|
| `"inner"` | 양쪽 표에 모두 있는 키의 행만 | 그 행이 사라진다 |
| `"left"` | 왼쪽 표의 행 전부 | 오른쪽 열이 `NaN`으로 채워진다 |

"주문 표는 그대로 두고 정보를 덧붙인다"가 목적이면 `how="left"`를 쓰고, 합친 뒤 `shape`으로 행 수가 바뀌지 않았는지 확인한다.

---

### **6. `merge()`와 `concat()`은 각각 언제 쓰는가?**

| 구분 | `merge()` | `concat()` |
|---|---|---|
| 기준 | 공통 열(키)의 값이 같은 행끼리 | 축(axis)을 따라 단순 연결 |
| 상황 | 작업 표에 담당자·검사 정보 덧붙이기 | 월별·라인별 파일을 하나로 합치기 |

위아래로 `concat()`할 때는 인덱스가 겹치지 않도록 `ignore_index=True`를 쓴다.

---

### **7. `df.groupby("지역")["매출"].mean()`은 어떤 순서로 계산되는가?**

1. `groupby("지역")`: 지역 값이 같은 행끼리 묶는다 (서울 묶음, 부산 묶음, 대전 묶음)
2. `["매출"]`: 각 묶음에서 매출 열만 꺼낸다
3. `.mean()`: 묶음마다 평균을 낸다

결과는 지역이 인덱스이고 평균이 값인 Series다. 여러 계산을 한 번에 하려면 `.agg(["sum", "mean", "count"])`를 쓴다.

---

### **8. `pivot_table()`에서 `aggfunc`를 생략하면 어떻게 계산되는가?**

평균(`"mean"`)으로 계산된다. 같은 칸에 여러 건이 들어가면 그 평균이 들어가므로, 합계가 필요하면 `aggfunc="sum"`을 꼭 적는다.

---

### **9. `melt()`와 `pivot_table()`은 어떤 관계인가?**

서로 반대 방향이다. `pivot_table()`은 세로로 긴 표를 행 × 열 격자로 펼치고(세로 → 가로), `melt()`는 여러 열에 흩어진 값을 세로로 모은다(가로 → 세로). `melt()`는 값을 계산하지 않고 모양만 바꾼다.

---
